# Assignment 4 — GDPR-Compliant Mini Hospital Management System

**Course:** Information Security (CS-3002)  
**Student:** Muddassir Rizwan (23K-0784)

This notebook demonstrates each component of the Hospital Privacy Dashboard step by step using the exact same modules (`crypto_utils.py`, `auth.py`, `database.py`) that power the Streamlit app (`app.py`).

It verifies all three pillars of the **CIA Triad** (Confidentiality, Integrity, Availability) along with the **GDPR** features (consent tracking, Fernet reversible anonymization, retention timer, and right to erasure).

## Step 1: Initialize Database Schema, Constraints & Sample Data

`init_db()` creates `hospital.db` with `users`, `patients`, `logs`, and `settings` tables, enables foreign keys, adds `CHECK` constraints, and attaches two append-only SQLite triggers (`prevent_log_update` and `prevent_log_delete`) so audit logs cannot be tampered with.

In [1]:
from database import get_all_patients, get_connection, init_db

init_db()

patients = get_all_patients()
print(f"Total sample patients: {len(patients)}")
for p in patients:
    print(f"  #{p[0]} | {p[1]} | {p[2]} | {p[3]} | {p[4]} | {p[5]} | {p[6]}")

conn = get_connection()
triggers = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='trigger'").fetchall()]
conn.close()
print("Active SQLite integrity triggers:", triggers)

Total sample patients: 8
  #1 | Ahmed Khan | 0300-1234567 | Flu | ANON_1796 | XXX-XXX-4567 | 2026-11-20
  #2 | Sara Ahmed | 0321-7654321 | Diabetes | ANON_5858 | XXX-XXX-4321 | 2026-11-21
  #3 | Bilal Hussain | 0333-4567890 | Fracture | ANON_9774 | XXX-XXX-7890 | 2026-11-22
  #4 | Ayesha Malik | 0345-9876543 | Asthma | ANON_3394 | XXX-XXX-6543 | 2026-11-23
  #5 | Usman Ali | 0301-2345678 | Fever | ANON_7318 | XXX-XXX-5678 | 2026-11-24
  #6 | Fatima Noor | 0322-3456789 | Migraine | ANON_1617 | XXX-XXX-6789 | 2026-11-25
  #7 | Hassan Raza | 0312-8877665 | Hypertension | ANON_8960 | XXX-XXX-7665 | 2025-12-15
  #8 | Zainab Tariq | 0331-5544332 | Seasonal Allergy | ANON_9749 | XXX-XXX-4332 | 2025-06-10
Active SQLite integrity triggers: ['prevent_log_update', 'prevent_log_delete']


## Step 2: Authentication (Salted PBKDF2) & RBAC Enforcement

Demo accounts from the assignment (`admin`, `Dr. Bob`, `Alice_recep`) are stored with salted PBKDF2-HMAC-SHA256 hashes (100,000 iterations). Role permissions are enforced both in the UI and directly inside `database.py`.

In [2]:
from auth import can_do, check_login
from database import anonymize_all, get_all_logs

for uname, pw in [("admin", "admin123"), ("Dr. Bob", "doc123"), ("Alice_recep", "rec123"), ("admin", "wrongpw")]:
    res = check_login(uname, pw)
    print(f"{uname:12s} -> {res['role'] if res else 'LOGIN FAILED'}")

print("\nRBAC summary:")
for r in ["admin", "doctor", "receptionist"]:
    print(f"  {r:12s} | raw={can_do(r, 'view_raw')} | diag={can_do(r, 'view_diagnosis')} | add={can_do(r, 'add')} | anonymize={can_do(r, 'anonymize')} | logs={can_do(r, 'view_logs')}")

# Backend RBAC check: Doctor attempting to open audit logs or anonymize must raise PermissionError
for fn_name, fn in [("get_all_logs", lambda: get_all_logs(role="doctor")), ("anonymize_all", lambda: anonymize_all(role="receptionist"))]:
    try:
        fn()
    except PermissionError as err:
        print(f"Blocked unauthorized {fn_name}: {err}")

admin        -> admin
Dr. Bob      -> doctor
Alice_recep  -> receptionist
admin        -> LOGIN FAILED

RBAC summary:
  admin        | raw=True | diag=True | add=True | anonymize=True | logs=True
  doctor       | raw=False | diag=True | add=False | anonymize=False | logs=False
  receptionist | raw=False | diag=False | add=True | anonymize=False | logs=False
Blocked unauthorized get_all_logs: Role 'doctor' is not authorized for 'view_logs'.
Blocked unauthorized anonymize_all: Role 'receptionist' is not authorized for 'anonymize'.


## Step 3: Confidentiality — Data Masking & Role-Scoped Views

Names are masked to deterministic `ANON_XXXX` IDs using SHA-256, contacts are masked to `XXX-XXX-4567`, and diagnoses are masked as `***` for Receptionist access. `get_patients_for_role()` ensures raw PII is never even sent to Doctor or Receptionist views.

In [3]:
from crypto_utils import make_anon_name, mask_contact, mask_diagnosis, mask_name
from database import get_patients_for_role

print("Masking helpers:")
print("  mask_name('Ahmed Khan')        ->", mask_name("Ahmed Khan"))
print("  make_anon_name('Ahmed Khan',1) ->", make_anon_name("Ahmed Khan", 1))
print("  mask_contact('0300-1234567')   ->", mask_contact("0300-1234567"))
print("  mask_diagnosis('Flu')          ->", mask_diagnosis("Flu"))

print("\nWhat each role receives for Patient #1:")
for r in ["admin", "doctor", "receptionist"]:
    row = get_patients_for_role(r)[0]
    print(f"  {r:12s} -> {row}")

Masking helpers:
  mask_name('Ahmed Khan')        -> ***
  make_anon_name('Ahmed Khan',1) -> ANON_1796
  mask_contact('0300-1234567')   -> XXX-XXX-4567
  mask_diagnosis('Flu')          -> ***

What each role receives for Patient #1:
  admin        -> {'patient_id': 1, 'name': 'Ahmed Khan', 'contact': '0300-1234567', 'diagnosis': 'Flu', 'anonymized_name': 'ANON_1796', 'anonymized_contact': 'XXX-XXX-4567', 'encrypted_preview': 'gAAAAABrCrCG2WBoxCh063...', 'date_added': '2026-11-20', 'days_old': 8, 'days_left': 357, 'retention_status': 'Active (357d left)', 'consent_given': True, 'is_erased': False}
  doctor       -> {'patient_id': 1, 'anonymized_name': 'ANON_1796', 'anonymized_contact': 'XXX-XXX-4567', 'diagnosis': 'Flu', 'date_added': '2026-11-20', 'days_old': 8, 'retention_status': 'Active (357d left)', 'is_erased': False}
  receptionist -> {'patient_id': 1, 'anonymized_name': 'ANON_1796', 'anonymized_contact': 'XXX-XXX-4567', 'diagnosis': '***', 'date_added': '2026-11-20', 'days_old':

## Step 4: Bonus — Reversible Anonymization with Fernet Encryption

When Admin triggers `anonymize_all()`, raw `name` and `contact` values inside `patients` are replaced with `ANON_XXXX` and `XXX-XXX-4567`, while Fernet ciphertexts are stored in `encrypted_name` and `encrypted_contact` using `secret.key`. Calling `decrypt_all()` decrypts those tokens back to the original plaintext values.

In [4]:
from crypto_utils import decrypt_text
from database import anonymize_all, decrypt_all, get_encrypted_sample

n_anon = anonymize_all(role="admin")
print(f"Anonymized & encrypted {n_anon} records in SQLite.")
sample = get_encrypted_sample()
print(f"DB row #1 while anonymized -> name='{sample[1]}', anon='{sample[2]}', contact='{sample[3]}'")
print(f"Fernet ciphertext          -> {sample[4][:60]}...")
print(f"Direct Fernet decrypt test -> {decrypt_text(sample[4])}")

n_dec = decrypt_all(role="admin")
restored = get_all_patients()[0]
print(f"Decrypted {n_dec} records -> DB row #1 restored: name='{restored[1]}', contact='{restored[2]}'")

Anonymized & encrypted 8 records in SQLite.
DB row #1 while anonymized -> name='ANON_1796', anon='ANON_1796', contact='XXX-XXX-4567'
Fernet ciphertext          -> gAAAAABrCrCGTtqNYU0AfchTIWjfE9zMPjO1Okyp_JyNTc6Q0HfTnEfw4aox...
Direct Fernet decrypt test -> Ahmed Khan
Decrypted 8 records -> DB row #1 restored: name='Ahmed Khan', contact='0300-1234567'


## Step 5: Integrity — Consent Validation, Blind Edit Sync & Immutable Audit Log

Here we verify:
1. Adding a patient without consent is blocked.
2. Receptionist blind-editing a patient's contact keeps `anonymized_contact` and `encrypted_contact` synced while leaving `diagnosis` untouched.
3. SQLite's `prevent_log_delete` trigger blocks any attempt to delete audit logs.

In [5]:
import sqlite3
from database import add_patient, get_all_logs, log_action, update_patient

# 1. Consent enforcement check
try:
    add_patient("No Consent", "0300-1112222", "Cold", "2026-11-28", consent_given=False, role="receptionist")
except ValueError as err:
    print("Consent block works:", err)

# 2. Add patient with consent + Receptionist blind edit (name blank, new contact, diagnosis ignored)
pid = add_patient("Test Patient", "0300-9991111", "Cold", "2026-11-28", consent_given=True, role="receptionist")
log_action(3, "receptionist", "add", f"Added patient #{pid} (consent verified)")
changed = update_patient(pid, name="", contact="0300-9992222", diagnosis="ShouldBeIgnored", role="receptionist")
log_action(3, "receptionist", "update", f"Updated patient #{pid} (fields: {', '.join(changed)})")

updated_row = [p for p in get_all_patients() if p[0] == pid][0]
print(f"After blind edit -> changed={changed}, contact='{updated_row[2]}', masked='{updated_row[5]}', diagnosis='{updated_row[3]}'")

# Clean up temporary test patient
conn = get_connection()
conn.execute("DELETE FROM patients WHERE patient_id = ?", (pid,))
conn.commit()

# 3. Verify SQLite trigger blocks audit log deletion
try:
    conn.execute("DELETE FROM logs")
except sqlite3.IntegrityError as err:
    print("Immutable log trigger blocked DELETE:", err)
finally:
    conn.close()

print("Latest audit log entries:")
for entry in get_all_logs(role="admin")[:3]:
    print(" ", entry)

Consent block works: Patient consent is required under GDPR.
After blind edit -> changed=['contact'], contact='0300-9992222', masked='XXX-XXX-2222', diagnosis='Cold'
Immutable log trigger blocked DELETE: Audit logs are immutable and cannot be deleted
Latest audit log entries:
  (2, 3, 'receptionist', 'update', '2026-11-28 15:44:22', 'Updated patient #9 (fields: contact)')
  (1, 3, 'receptionist', 'add', '2026-11-28 15:44:22', 'Added patient #9 (consent verified)')


## Step 6: Availability & GDPR Retention Timer

Admin can export CSV backups of both patient records (raw or anonymized) and audit logs. Each patient record tracks its age against the GDPR retention limit (default 365 days) so expired records are clearly flagged for review or PII erasure.

In [6]:
from database import export_logs_csv, export_patients_csv, get_actions_per_day, get_retention_days

print("Patients CSV (anonymized) preview:")
for line in export_patients_csv(anonymized=True, role="admin").splitlines()[:3]:
    print(" ", line)

print("\nAudit Logs CSV header:", export_logs_csv(role="admin").splitlines()[0])

print(f"\nGDPR Retention Status (policy = {get_retention_days()} days):")
for p in get_patients_for_role("admin"):
    print(f"  #{p['patient_id']} {p['anonymized_name']} | added {p['date_added']} | {p['days_old']:3d}d old | {p['retention_status']}")

print("\nActions per day:", get_actions_per_day(role="admin"))

Patients CSV (anonymized) preview:
  patient_id,name,contact,diagnosis,anonymized_name,anonymized_contact,date_added
  1,ANON_1796,XXX-XXX-4567,Flu,ANON_1796,XXX-XXX-4567,2026-11-20
  2,ANON_5858,XXX-XXX-4321,Diabetes,ANON_5858,XXX-XXX-4321,2026-11-21

Audit Logs CSV header: log_id,user_id,role,action,timestamp,details

GDPR Retention Status (policy = 365 days):
  #1 ANON_1796 | added 2026-11-20 |   8d old | Active (357d left)
  #2 ANON_5858 | added 2026-11-21 |   7d old | Active (358d left)
  #3 ANON_9774 | added 2026-11-22 |   6d old | Active (359d left)
  #4 ANON_3394 | added 2026-11-23 |   5d old | Active (360d left)
  #5 ANON_7318 | added 2026-11-24 |   4d old | Active (361d left)
  #6 ANON_1617 | added 2026-11-25 |   3d old | Active (362d left)
  #7 ANON_8960 | added 2025-12-15 | 348d old | Expiring Soon (17d left)
  #8 ANON_9749 | added 2025-06-10 | 536d old | Expired (171d over)

Actions per day: [('2026-11-28', 'add', 1), ('2026-11-28', 'update', 1)]
